In [ ]:
import os
import pandas as pd
import SimpleITK as sitk
import logging
import numpy as np
from radiomics import featureextractor

# --- 1. Configuration & Initialization ---
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

BASE_DIR = r"./dataimage"
OUTPUT_CSV = "radiomics_features_EV_standardized.csv"

settings = {
    'binWidth': 25,
    'resampledPixelSpacing': [1, 1, 1],
    'interpolator': sitk.sitkBSpline,
    'geometryTolerance': 1e-3,
    'correctMask': True,
    'enableCExtensions': True
}

extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
feature_classes = ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']
for cls in feature_classes:
    extractor.enableFeatureClassByName(cls)

def extract_features(image_obj, mask_obj, label_id, prefix):
    try:
        extractor.settings['label'] = label_id
        feature_vector = extractor.execute(image_obj, mask_obj)
        return {f"{prefix}_{k}": v for k, v in feature_vector.items() if not k.startswith("diagnostics")}
    except Exception as e:
        return {}

# --- 2. Feature Extraction Loop ---
data_list = []
classes = ['bleeding', 'non_bleeding']

print(">>> Starting Radiomics Extraction...")
for class_label in classes:
    class_path = os.path.join(BASE_DIR, class_label)
    if not os.path.exists(class_path): continue
        
    patient_folders = [f for f in os.listdir(class_path) if os.path.isdir(os.path.join(class_path, f))]
    
    for patient_id in patient_folders:
        patient_dir = os.path.join(class_path, patient_id)
        vol_path = os.path.join(patient_dir, "volume.nii.gz")
        target_path = os.path.join(patient_dir, "target.nii.gz")
        varix_path = os.path.join(patient_dir, "varix.nii.gz")
        
        if not (os.path.exists(vol_path) and os.path.exists(target_path) and os.path.exists(varix_path)):
            continue
            
        try:
            sitk_vol = sitk.ReadImage(vol_path)
            sitk_target = sitk.ReadImage(target_path)
            sitk_varix = sitk.ReadImage(varix_path)
            
            patient_features = {'PatientID': patient_id, 'Label': 1 if class_label == 'bleeding' else 0}
            patient_features.update(extract_features(sitk_vol, sitk_target, 1, "Spleen"))
            patient_features.update(extract_features(sitk_vol, sitk_target, 2, "Liver"))
            patient_features.update(extract_features(sitk_vol, sitk_target, 3, "Esophagus"))
            patient_features.update(extract_features(sitk_vol, sitk_varix, 1, "Varix"))
            
            data_list.append(patient_features)
        except Exception:
            continue

if data_list:
    df = pd.DataFrame(data_list)
    cols = ['PatientID', 'Label'] + [c for c in df.columns if c not in ['PatientID', 'Label']]
    df[cols].to_csv(OUTPUT_CSV, index=False)
    print(f"Extraction completed. Features saved to {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import numpy as np
import random
import collections
from scipy.stats import mannwhitneyu
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, roc_curve

# --- 1. Reproducibility & Data Loading ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(">>> Step 1: Loading Internal Data...")
df_rad = pd.read_csv("radiomics_features_EV_standardized.csv")
df_rad['PatientID'] = df_rad['PatientID'].astype(str).str.replace(r'\.0$', '', regex=True)

try:
    xls = pd.read_excel("clinical_data.xlsx", sheet_name=None)
    df_clin = pd.concat([d for _, d in xls.items()], ignore_index=True)
    df_clin.columns = [c.strip().lower() for c in df_clin.columns]
    id_col = [c for c in df_clin.columns if 'id' in c][0]
    df_clin.rename(columns={id_col: 'PatientID'}, inplace=True)
    df_clin['PatientID'] = df_clin['PatientID'].astype(str).str.replace(r'\.0$', '', regex=True)
    
    df_master = pd.merge(df_rad, df_clin, on='PatientID', how='inner')
    if 'pvt' not in df_master.columns: df_master['pvt'] = 0
except Exception as e:
    print(f"Clinical data load warning: {e}")
    df_master = df_rad.copy()

y = df_master['Label'].values

# --- 2. Definition of Modeling Scenarios ---
all_cols = df_master.columns.tolist()
varix_cols = [c for c in all_cols if str(c).startswith('Varix')]
organ_cols = [c for c in all_cols if any(k in c for k in ['Spleen', 'Liver', 'Esophagus'])]
clin_cols = [c for c in ['age', 'gender', 'plt', 'inr', 'ast', 'alt', 'albumin', 'pvt'] if c in df_master.columns]

SCENARIOS = {
    "Model A (Organ+Varix)": varix_cols + organ_cols,
    "Model B (Varix Only)": varix_cols,
    "Model C (Organ Only)": organ_cols,
    "Model D (Clinical)": clin_cols,
    "Model E (Varix+Clin)": varix_cols + clin_cols,
    "Model F (Ultimate)": varix_cols + organ_cols + clin_cols
}

# --- 3. Feature Selection & Bootstrapping Utilities ---
def select_features_internal(X_train, y_train, feature_names):
    if not feature_names: return []
    X_df = pd.DataFrame(X_train, columns=feature_names)
    y_arr = np.array(y_train)
    
    pass_mw = []
    for col in feature_names:
        g0, g1 = X_df.loc[y_arr==0, col].dropna(), X_df.loc[y_arr==1, col].dropna()
        if len(g0) < 2 or len(g1) < 2: continue
        try:
            if mannwhitneyu(g0, g1)[1] < 0.05: pass_mw.append(col)
        except: continue
            
    if not pass_mw: return []
    
    X_scaled = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X_df[pass_mw].values))
    lasso = LogisticRegressionCV(Cs=5, cv=3, penalty='l1', solver='liblinear', class_weight='balanced', random_state=SEED, max_iter=1000)
    try:
        lasso.fit(X_scaled, y_arr)
        return [pass_mw[i] for i in np.where(np.abs(lasso.coef_[0]) > 1e-5)[0]]
    except:
        return pass_mw[:5]

def compute_metrics_with_ci(y_true, y_probs, threshold, n_bootstraps=1000):
    y_true, y_probs = np.array(y_true), np.array(y_probs)
    y_pred = (y_probs >= threshold).astype(int)
    auc_score = roc_auc_score(y_true, y_probs)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    auc_boots, sens_boots, spec_boots = [], [], []
    rng = np.random.RandomState(SEED)
    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_probs), len(y_probs))
        if len(np.unique(y_true[idx])) < 2: continue
        auc_boots.append(roc_auc_score(y_true[idx], y_probs[idx]))
        tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_true[idx], y_pred[idx]).ravel()
        sens_boots.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
        spec_boots.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)

    def get_ci(scores): return np.sort(scores)[int(0.025 * len(scores))], np.sort(scores)[int(0.975 * len(scores))]
    return {"AUC": (auc_score, *get_ci(auc_boots)), "Sens": (tp/(tp+fn), *get_ci(sens_boots)), "Spec": (tn/(tn+fp), *get_ci(spec_boots))}

# --- 4. Nested Cross-Validation with ALL Machine Learning Settings ---
print(">>> Step 2: Training & Evaluation (Nested CV)...")

classifiers = {
    "Lasso": LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced', random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=SEED),
    "GBM": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=SEED)
}

final_results_list = []
feature_stability_tracker = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for s_name, candidate_feats in SCENARIOS.items():
    if not candidate_feats: continue
    feature_stability_tracker[s_name] = []
    scenario_preds = {algo: {'probs': np.zeros(len(y)), 'indices': []} for algo in classifiers}
    X_full_df = df_master[candidate_feats]
    
    for train_idx, val_idx in cv.split(X_full_df, y):
        X_train, X_val = X_full_df.iloc[train_idx], X_full_df.iloc[val_idx]
        selected_feats = select_features_internal(X_train, y[train_idx], candidate_feats)
        feature_stability_tracker[s_name].extend(selected_feats)
        
        if not selected_feats: continue
        
        pipeline_pre = Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())])
        X_train_trans = pipeline_pre.fit_transform(X_train[selected_feats])
        X_val_trans = pipeline_pre.transform(X_val[selected_feats])
        
        for clf_name, clf in classifiers.items():
            clf.fit(X_train_trans, y[train_idx])
            scenario_preds[clf_name]['probs'][val_idx] = clf.predict_proba(X_val_trans)[:, 1]

    for clf_name, preds_data in scenario_preds.items():
        final_probs = preds_data['probs']
        fpr, tpr, thres = roc_curve(y, final_probs)
        best_th = thres[np.argmax(tpr - fpr)]
        metrics = compute_metrics_with_ci(y, final_probs, best_th)
        final_results_list.append({
            "Scenario": s_name, "Algo": clf_name, 
            "AUC_Raw": metrics['AUC'][0],
            "AUC": f"{metrics['AUC'][0]:.3f} ({metrics['AUC'][1]:.3f}-{metrics['AUC'][2]:.3f})",
            "Sens": f"{metrics['Sens'][0]:.3f} ({metrics['Sens'][1]:.3f}-{metrics['Sens'][2]:.3f})",
            "Spec": f"{metrics['Spec'][0]:.3f} ({metrics['Spec'][1]:.3f}-{metrics['Spec'][2]:.3f})"
        })

df_display = pd.DataFrame(final_results_list)[['Scenario', 'Algo', 'AUC', 'Sens', 'Spec']]
print("\n=== Model Performance Summary (Internal) ===")
print(df_display.to_string(index=False))

# --- 5. Export Selected Features ---
final_selection_records = []
for model_name, raw_features in feature_stability_tracker.items():
    if not raw_features: continue
    counts = collections.Counter(raw_features)
    robust_features = [feat for feat, freq in counts.items() if freq >= 3]
    if not robust_features: robust_features = [feat for feat, _ in counts.most_common(3)]
    final_selection_records.append({"Model": model_name, "Features": ",".join(robust_features)})

pd.DataFrame(final_selection_records).to_csv("Selected_Features_List.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss, calibration_curve

print(">>> Step 3: External Validation & Advanced Analytics...")

# Load external dataset (Modify paths as needed for local execution)
# df_ext = pd.read_csv("External_Features_Final.csv")

base_classifiers = {
    "Lasso": LogisticRegressionCV(Cs=10, cv=5, penalty='l1', solver='liblinear', class_weight='balanced', random_state=42, max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    "GBM": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

df_feats = pd.read_csv("Selected_Features_List.csv")
FEATURE_SETS = {row['Model']: row['Features'].split(',') for _, row in df_feats.iterrows()}

best_models_plot = {} 
shap_models = {} 
brier_list = []

for s_name, feats in FEATURE_SETS.items():
    if not feats: continue
    
    for f in feats:
        if f not in df_ext.columns: df_ext[f] = 0
        if f not in df_int.columns: df_int[f] = 0
            
    X_train = df_int[feats].fillna(df_int[feats].median())
    X_test = df_ext[feats].fillna(df_int[feats].median())
    
    scenario_res = []
    
    for clf_name, base_clf in base_classifiers.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('clf', clone(base_clf))])
        pipe.fit(X_train, y_int)
        
        probs_train = pipe.predict_proba(X_train)[:, 1]
        fpr, tpr, thres = roc_curve(y_int, probs_train)
        best_th = thres[np.argmax(tpr - fpr)]
        
        probs_ext = pipe.predict_proba(X_test)[:, 1]
        metrics = compute_metrics_with_ci(y_ext, probs_ext, best_th)
        
        bs = brier_score_loss(y_ext, probs_ext)
        brier_list.append({"Model": s_name, "Algo": clf_name, "AUC": metrics['AUC'][0], "Brier Score": bs})
        
        res_data = {
            "Model": s_name, "Algo": clf_name, "AUC_Raw": metrics['AUC'][0],
            "Probs": probs_ext, "Pipeline": pipe, "X_train": X_train
        }
        scenario_res.append(res_data)

    best_entry = sorted(scenario_res, key=lambda x: x['AUC_Raw'], reverse=True)[0]
    best_models_plot[s_name] = best_entry
    shap_models[s_name] = best_entry 

# --- Output Brier Score Summary ---
print("\n=== Brier Score Summary ===")
df_brier = pd.DataFrame(brier_list).sort_values(by="AUC", ascending=False)
print(df_brier[['Model', 'Algo', 'Brier Score']].head(10).to_string(index=False))

# --- Decision Curve Analysis (DCA) ---
def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        net_benefits.append((tp / n) - (fp / n) * (thresh / (1 - thresh)))
    return np.array(net_benefits)

plt.figure(figsize=(8, 6))
thresholds = np.linspace(0.01, 0.99, 100)
prevalence = np.mean(y_ext)
plt.plot(thresholds, prevalence - (1-prevalence)*thresholds/(1-thresholds), ':', label='Treat All', color='gray')
plt.plot([0, 1], [0, 0], '--', label='Treat None', color='black')

colors = ['purple', 'green', 'orange', 'blue', 'brown', 'red']
for idx, (name, res) in enumerate(sorted(best_models_plot.items())):
    net_benefit = calculate_net_benefit(y_ext, res['Probs'], thresholds)
    plt.plot(thresholds, net_benefit, label=name, color=colors[idx % len(colors)], lw=2)

plt.xlim(0, 1); plt.ylim(-0.05, 0.5)
plt.title("Decision Curve Analysis (External)", fontsize=16)
plt.xlabel("Threshold Probability"); plt.ylabel("Net Benefit")
plt.legend()
plt.savefig("External_DCA.png", dpi=300)
plt.show()

# --- SHAP Interpretability Analysis ---
print("\n>>> Generating SHAP Plots...")
for name in ["Model E (Varix+Clin)", "Model B (Varix Only)", "Model F (Ultimate)"]:
    if name not in shap_models: continue
    model_entry = shap_models[name]
    clf, scaler = model_entry['Pipeline'].named_steps['clf'], model_entry['Pipeline'].named_steps['scaler']
    
    X_scaled = pd.DataFrame(scaler.transform(model_entry['X_train']), columns=model_entry['X_train'].columns)
    
    try:
        if isinstance(clf, (RandomForestClassifier, GradientBoostingClassifier)):
            explainer = shap.TreeExplainer(clf)
            shap_values = explainer.shap_values(X_scaled, check_additivity=False)
            if isinstance(shap_values, list): shap_values = shap_values[1] 
        elif isinstance(clf, (LogisticRegression, LogisticRegressionCV)):
            explainer = shap.LinearExplainer(clf, X_scaled)
            shap_values = explainer.shap_values(X_scaled)

        plt.figure()
        plt.title(f"SHAP Summary: {name}")
        shap.summary_plot(shap_values, X_scaled, show=False)
        plt.tight_layout()
        plt.savefig(f"SHAP_{name.replace(' ', '_')}.png", dpi=300)
        plt.show()
    except Exception as e:
        print(f"SHAP failed for {name}: {e}")